# MLP Default Prediction

This notebook trains a multilayer perceptron to predict `default payment next month` from the credit-card client dataset.

Model choice: for this 30,000-row tabular dataset, a medium MLP is a good neural baseline. Very large neural networks usually do not help much on tabular data and can overfit quickly. Strong classical baselines such as logistic regression, random forests, XGBoost/LightGBM/CatBoost are often competitive or better, but this notebook focuses on a well-sized MLP.

## GPU Check

In this current shell, `nvidia-smi` failed with: NVIDIA-SMI could not communicate with the NVIDIA driver. That means no usable NVIDIA GPU is visible from this session right now.

The notebook below automatically uses CUDA if PyTorch can see a GPU; otherwise it uses CPU. For this dataset, CPU training should still be fast because the data is small.

## Required Packages

Install these in the `fairfed` environment if needed:

```bash
conda activate fairfed
conda install scikit-learn matplotlib
pip install torch
```

If your machine has a CUDA GPU, install the PyTorch build matching your CUDA version from the official PyTorch install selector.


In [1]:
from pathlib import Path
from itertools import product
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        print(idx, torch.cuda.get_device_name(idx))

try:
    from credit_data import prepare_feature_sets
except ModuleNotFoundError:
    import sys

    sys.path.append(str(Path.cwd() / "src"))
    from credit_data import prepare_feature_sets


Device: cuda
CUDA device count: 1
0 NVIDIA A40


In [2]:
DATA_PATHS = [
    Path("../data/default_of_credit_card_clients.xls"),
    Path("data/default_of_credit_card_clients.xls"),
]
DATA_PATH = next(path for path in DATA_PATHS if path.exists())

df = pd.read_excel(DATA_PATH, header=1)
target_col = "default payment next month"

print(DATA_PATH)
print(df.shape)
df.head()


../data/default_of_credit_card_clients.xls
(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## Feature Plan

This notebook compares two versions of the input data for the MLP: the original credit-card features and a second version with extra behavioral summaries. `ID` is removed because it only labels each row and should not help the model predict default risk.

The engineered features summarize repayment delays, bill amounts, payment amounts, credit-limit utilization, and recent bill changes. These features are added to make repeated monthly patterns easier for the model to learn.

Categorical columns are one-hot encoded so the network can use them as separate indicator variables, while numeric columns are standardized because MLPs train more reliably when inputs are on similar scales. Both feature sets use the same stratified train, validation, and test split so any performance difference is due to the features rather than different data partitions.


In [3]:
feature_sets = prepare_feature_sets(df, target_col=target_col, seed=SEED)

ACTIVE_FEATURE_SET = "engineered"
active_data = feature_sets[ACTIVE_FEATURE_SET]

X_train = active_data["X_train"]
X_val = active_data["X_val"]
X_test = active_data["X_test"]
y_train = active_data["y_train"]
y_val = active_data["y_val"]
y_test = active_data["y_test"]
numeric_cols = active_data["numeric_cols"]
preprocess = active_data["preprocess"]

for name, data_bundle in feature_sets.items():
    print(
        f"{name}: Train {data_bundle['X_train'].shape}, "
        f"Val {data_bundle['X_val'].shape}, Test {data_bundle['X_test'].shape}, "
        f"engineered={data_bundle['use_feature_engineering']}"
    )
print("Active feature set:", ACTIVE_FEATURE_SET)
print("Default rate train:", y_train.mean().round(4))


baseline: Train (21000, 91), Val (4500, 91), Test (4500, 91), engineered=False
engineered: Train (21000, 105), Val (4500, 105), Test (4500, 105), engineered=True
Active feature set: engineered
Default rate train: 0.2212


In [4]:
def make_dataset(X_array, y_series):
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    y_tensor = torch.tensor(y_series.to_numpy(), dtype=torch.float32).view(-1, 1)
    return TensorDataset(X_tensor, y_tensor)


def activate_feature_set(feature_set_name):
    global active_data, X_train, X_val, X_test, y_train, y_val, y_test, numeric_cols, preprocess

    active_data = feature_sets[feature_set_name]
    X_train = active_data["X_train"]
    X_val = active_data["X_val"]
    X_test = active_data["X_test"]
    y_train = active_data["y_train"]
    y_val = active_data["y_val"]
    y_test = active_data["y_test"]
    numeric_cols = active_data["numeric_cols"]
    preprocess = active_data["preprocess"]
    return active_data


batch_size = 512
activate_feature_set(ACTIVE_FEATURE_SET)

train_loader = DataLoader(make_dataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(make_dataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(make_dataset(X_test, y_test), batch_size=batch_size, shuffle=False)


## Hyperparameter Search

Accuracy is not enough for this imbalanced binary problem. This section tries several MLP configurations and selects models based on validation metrics for the `default = 1` class.

The most important metrics here are:

- `val_f1`: balance between default precision and default recall
- `val_recall`: how many true defaults are found
- `val_precision`: how many predicted defaults are actually defaults
- `val_roc_auc`: ranking quality across thresholds
- `val_pr_auc`: precision-recall quality for the minority class


In [5]:
class TunableCreditMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.2, use_batch_norm=False):
        super().__init__()
        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_loader(X_array, y_series, batch_size, shuffle):
    return DataLoader(make_dataset(X_array, y_series), batch_size=batch_size, shuffle=shuffle)


def probabilities_for_model(model_to_eval, loader):
    model_to_eval.eval()
    probs = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model_to_eval(xb.to(device))
            probs.append(torch.sigmoid(logits).cpu().numpy().ravel())
            targets.append(yb.numpy().ravel())
    return np.concatenate(targets), np.concatenate(probs)


def best_threshold_metrics(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
        })
    metrics = pd.DataFrame(rows)
    best = metrics.sort_values(["f1", "recall"], ascending=False).iloc[0]
    return best


In [6]:
def train_one_config(config, data_bundle, max_epochs=25, patience=5, verbose=False):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    X_train_cfg = data_bundle["X_train"]
    X_val_cfg = data_bundle["X_val"]
    y_train_cfg = data_bundle["y_train"]
    y_val_cfg = data_bundle["y_val"]

    train_loader_cfg = make_loader(X_train_cfg, y_train_cfg, config["batch_size"], shuffle=True)
    val_loader_cfg = make_loader(X_val_cfg, y_val_cfg, config["batch_size"], shuffle=False)

    candidate = TunableCreditMLP(
        input_dim=X_train_cfg.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
        use_batch_norm=config["batch_norm"],
    ).to(device)

    positive_count = float(y_train_cfg.sum())
    negative_count = float(len(y_train_cfg) - y_train_cfg.sum())
    pos_weight_local = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)

    if config["use_pos_weight"]:
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_local)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    opt = torch.optim.AdamW(candidate.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_score = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(1, max_epochs + 1):
        candidate.train()
        for xb, yb in train_loader_cfg:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = candidate(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        y_val_true_cfg, y_val_prob_cfg = probabilities_for_model(candidate, val_loader_cfg)
        val_auc = roc_auc_score(y_val_true_cfg, y_val_prob_cfg)
        threshold_metrics = best_threshold_metrics(y_val_true_cfg, y_val_prob_cfg)
        score = threshold_metrics["f1"]

        if verbose:
            print(f"epoch {epoch:02d} val_auc={val_auc:.4f} val_f1={score:.4f}")

        if score > best_score:
            best_score = score
            best_state = {key: value.detach().cpu().clone() for key, value in candidate.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    candidate.load_state_dict(best_state)
    y_val_true_cfg, y_val_prob_cfg = probabilities_for_model(candidate, val_loader_cfg)
    threshold_metrics = best_threshold_metrics(y_val_true_cfg, y_val_prob_cfg)

    return {
        **config,
        "epochs": epoch,
        "input_dim": X_train_cfg.shape[1],
        "best_threshold": float(threshold_metrics["threshold"]),
        "val_f1": float(threshold_metrics["f1"]),
        "val_precision": float(threshold_metrics["precision"]),
        "val_recall": float(threshold_metrics["recall"]),
        "val_roc_auc": float(roc_auc_score(y_val_true_cfg, y_val_prob_cfg)),
        "val_pr_auc": float(average_precision_score(y_val_true_cfg, y_val_prob_cfg)),
        "state_dict": best_state,
    }


### Search Grid

This grid is intentionally modest so it can run on CPU. Increase it later if you want a slower but broader search.

In [7]:
search_space = {
    "hidden_dims": [(64, 32), (128, 64), (256, 128, 64)],
    "dropout": [0.1, 0.2, 0.3],
    "batch_norm": [False, True],
    "batch_size": [128, 512],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-4],
    "use_pos_weight": [True],
}

configs = [
    dict(zip(search_space.keys(), values))
    for values in product(*search_space.values())
]

FEATURE_SETS_TO_COMPARE = ["baseline", "engineered"]
print(f"Training {len(configs)} configurations for each of {FEATURE_SETS_TO_COMPARE}")

search_results = []
best_result = None

for feature_set_name in FEATURE_SETS_TO_COMPARE:
    data_bundle = feature_sets[feature_set_name]
    print(f"\nFeature set: {feature_set_name} | input_dim={data_bundle['X_train'].shape[1]}")

    for idx, config in enumerate(configs, start=1):
        print(f"[{idx:02d}/{len(configs)}] {config}")
        result = train_one_config(config, data_bundle, max_epochs=25, patience=5)
        result["feature_set"] = feature_set_name
        result["use_feature_engineering"] = data_bundle["use_feature_engineering"]
        search_results.append(result)
        print(
            f"    val_f1={result['val_f1']:.4f} "
            f"precision={result['val_precision']:.4f} "
            f"recall={result['val_recall']:.4f} "
            f"auc={result['val_roc_auc']:.4f} "
            f"threshold={result['best_threshold']:.3f}"
        )

        if best_result is None or result["val_f1"] > best_result["val_f1"]:
            best_result = result

search_results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "state_dict"}
    for result in search_results
]).sort_values(["val_f1", "val_recall", "val_roc_auc"], ascending=False)

comparison_df = (
    search_results_df
    .sort_values(["feature_set", "val_f1", "val_recall", "val_roc_auc"], ascending=[True, False, False, False])
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values(["val_f1", "val_recall", "val_roc_auc"], ascending=False)
)

comparison_df[[
    "feature_set",
    "input_dim",
    "val_f1",
    "val_precision",
    "val_recall",
    "val_roc_auc",
    "val_pr_auc",
    "best_threshold",
    "hidden_dims",
    "dropout",
    "batch_norm",
    "batch_size",
    "lr",
]]


Training 72 configurations for each of ['baseline', 'engineered']

Feature set: baseline | input_dim=91
[01/72] {'hidden_dims': (64, 32), 'dropout': 0.1, 'batch_norm': False, 'batch_size': 128, 'lr': 0.001, 'weight_decay': 0.0001, 'use_pos_weight': True}


KeyboardInterrupt: 

In [ ]:
ignored_result_keys = {
    "state_dict",
    "epochs",
    "input_dim",
    "best_threshold",
    "val_f1",
    "val_precision",
    "val_recall",
    "val_roc_auc",
    "val_pr_auc",
    "feature_set",
    "use_feature_engineering",
}
best_config = {key: value for key, value in best_result.items() if key not in ignored_result_keys}
best_feature_set = best_result["feature_set"]
best_data = feature_sets[best_feature_set]

print("Best feature set:", best_feature_set)
print("Best config:")
print(best_config)
print("Best validation threshold:", best_result["best_threshold"])

best_model = TunableCreditMLP(
    input_dim=best_data["X_train"].shape[1],
    hidden_dims=best_config["hidden_dims"],
    dropout=best_config["dropout"],
    use_batch_norm=best_config["batch_norm"],
).to(device)
best_model.load_state_dict(best_result["state_dict"])

test_loader_best = make_loader(best_data["X_test"], best_data["y_test"], best_config["batch_size"], shuffle=False)
y_test_true_best, y_test_prob_best = probabilities_for_model(best_model, test_loader_best)
y_test_pred_best = (y_test_prob_best >= best_result["best_threshold"]).astype(int)

print("Test ROC-AUC:", round(roc_auc_score(y_test_true_best, y_test_prob_best), 4))
print("Test PR-AUC:", round(average_precision_score(y_test_true_best, y_test_prob_best), 4))
print("Test accuracy:", round(accuracy_score(y_test_true_best, y_test_pred_best), 4))
print("Test F1:", round(f1_score(y_test_true_best, y_test_pred_best), 4))
print("Test precision:", round(precision_score(y_test_true_best, y_test_pred_best), 4))
print("Test recall:", round(recall_score(y_test_true_best, y_test_pred_best), 4))
print()
print(classification_report(y_test_true_best, y_test_pred_best, target_names=["no default", "default"]))
print("Confusion matrix:")
print(confusion_matrix(y_test_true_best, y_test_pred_best))


### Faster or Larger Search

If this is too slow, reduce the grid to fewer values or lower `max_epochs`. If you want a stronger search, add more learning rates such as `2e-3` and `1e-4`, more batch sizes such as `64` and `256`, and try `use_pos_weight=False` to see whether threshold tuning alone works better.

## Model

This is intentionally not huge. The dataset has only 30k rows, so a network like `256 -> 128 -> 64 -> 32` is already expressive enough. Batch normalization and dropout help reduce overfitting.

In [ ]:
# Pick which feature set to use for the fixed-size MLP below.
# Use "baseline" for the original features or "engineered" for original + engineered features.
ACTIVE_FEATURE_SET = "engineered"
activate_feature_set(ACTIVE_FEATURE_SET)

train_loader = DataLoader(make_dataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(make_dataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(make_dataset(X_test, y_test), batch_size=batch_size, shuffle=False)

print("Training fixed MLP with feature set:", ACTIVE_FEATURE_SET)
print("Input dimension:", X_train.shape[1])


class CreditDefaultMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


model = CreditDefaultMLP(input_dim=X_train.shape[1]).to(device)
model


In [ ]:
positive_count = float(y_train.sum())
negative_count = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

print("pos_weight:", pos_weight.item())


pos_weight: 3.5209903717041016


In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    all_probs = []
    all_targets = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        all_probs.append(probs)
        all_targets.append(yb.detach().cpu().numpy().ravel())

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)
    avg_loss = total_loss / len(loader.dataset)
    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    return avg_loss, auc, pr_auc


def predict_proba(loader):
    model.eval()
    probs = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            probs.append(torch.sigmoid(logits).cpu().numpy().ravel())
            targets.append(yb.numpy().ravel())
    return np.concatenate(targets), np.concatenate(probs)


In [ ]:
max_epochs = 60
patience = 8
best_val_auc = -np.inf
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, max_epochs + 1):
    train_loss, train_auc, train_pr_auc = run_epoch(train_loader, training=True)
    val_loss, val_auc, val_pr_auc = run_epoch(val_loader, training=False)
    scheduler.step(val_auc)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_auc": train_auc,
        "train_pr_auc": train_pr_auc,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "val_pr_auc": val_pr_auc,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} auc {train_auc:.4f} pr_auc {train_pr_auc:.4f} | "
        f"val loss {val_loss:.4f} auc {val_auc:.4f} pr_auc {val_pr_auc:.4f}"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping after {epoch} epochs.")
        break

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_auc"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_auc"], label="validation")
axes[1].set_title("ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

fig.tight_layout()
plt.show()

## Choose a Threshold

The model outputs probabilities. Because defaults are the minority class, `0.5` is not always the best threshold. Here we choose the validation threshold that maximizes F1 score.

In [ ]:
y_val_true, y_val_prob = predict_proba(val_loader)
thresholds = np.linspace(0.05, 0.95, 181)
f1_scores = [f1_score(y_val_true, y_val_prob >= threshold) for threshold in thresholds]
best_threshold = thresholds[int(np.argmax(f1_scores))]

print("Best validation F1 threshold:", round(float(best_threshold), 3))
print("Best validation F1:", round(float(max(f1_scores)), 4))


In [ ]:
y_test_true, y_test_prob = predict_proba(test_loader)
y_test_pred = (y_test_prob >= best_threshold).astype(int)

print("Test ROC-AUC:", round(roc_auc_score(y_test_true, y_test_prob), 4))
print("Test PR-AUC:", round(average_precision_score(y_test_true, y_test_prob), 4))
print("Test accuracy:", round(accuracy_score(y_test_true, y_test_pred), 4))
print("Test F1:", round(f1_score(y_test_true, y_test_pred), 4))
print()
print(classification_report(y_test_true, y_test_pred, target_names=["no default", "default"]))
print("Confusion matrix:")
print(confusion_matrix(y_test_true, y_test_pred))


In [ ]:
fpr, tpr, _ = roc_curve(y_test_true, y_test_prob)
precision, recall, _ = precision_recall_curve(y_test_true, y_test_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

axes[1].plot(recall, precision)
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

fig.tight_layout()
plt.show()

NameError: name 'y_test_true' is not defined

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_artifact = {
    "model_state_dict": model.state_dict(),
    "input_dim": X_train.shape[1],
    "best_threshold": float(best_threshold),
    "feature_set": ACTIVE_FEATURE_SET,
    "use_feature_engineering": active_data["use_feature_engineering"],
    "engineered_cols": active_data["engineered_cols"],
    "categorical_cols": active_data["categorical_cols"],
    "numeric_cols": active_data["numeric_cols"],
}

torch.save(model_artifact, MODEL_DIR / "credit_default_mlp.pt")

print("Saved model to", MODEL_DIR / "credit_default_mlp.pt")
print("Saved feature set:", ACTIVE_FEATURE_SET)
